# P10 - Estadistica consolidada

Consolida P7-1, P8-1 y P9. Las pruebas inferenciales se realizan **solo** sobre las 20 semillas pareadas de P9. Los resultados clasicos de P7/P8 se integran de forma descriptiva porque existe una sola evaluacion final por modelo y escenario; los folds de CV no se consideran corridas independientes.

In [1]:
from pathlib import Path
from itertools import combinations
import warnings
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# ============================================================
# P10 - Estadistica consolidada
# ============================================================
BASE_DIR = Path(r'C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
OUT_DIR = BASE_DIR / 'P10_estadistica_consolidada'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'p7_final': BASE_DIR / 'P7_desbalance_clases/P7-1_Desbalance_FINAL_test_metrics.csv',
    'p8_final': BASE_DIR / 'P8_hard_negatives/P8-1_HardNegatives_FINAL_test_metrics.csv',
    'p9_runs': BASE_DIR / 'P9_gnn_temporal_multiseed/P9_GNN_FINAL_test_runs.csv',
    'p9_summary': BASE_DIR / 'P9_gnn_temporal_multiseed/P9_GNN_FINAL_test_summary.csv',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos requeridos para P10:\n' + '\n'.join(missing))

P7 = pd.read_csv(FILES['p7_final'])
P8 = pd.read_csv(FILES['p8_final'])
P9 = pd.read_csv(FILES['p9_runs'])
P9S = pd.read_csv(FILES['p9_summary'])

METRICS_GNN = ['auc_roc', 'auc_pr', 'f1', 'accuracy', 'balanced_accuracy', 'precision', 'recall']
SEEDS = sorted(P9['seed'].dropna().astype(int).unique().tolist())
MODELS_GNN = sorted(P9['model'].dropna().unique().tolist())

# -----------------------------
# Utilidades
# -----------------------------
def bootstrap_mean_ci(x, n_boot=10000, alpha=0.05, seed=20261009):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    means = x[idx].mean(axis=1)
    lo, hi = np.quantile(means, [alpha/2, 1-alpha/2])
    return float(lo), float(hi)


def iqr(x):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    if len(x) == 0:
        return np.nan
    return float(np.quantile(x, .75) - np.quantile(x, .25))


def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adjusted_sorted = np.empty(m, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        running = max(running, val)
        adjusted_sorted[rank] = min(running, 1.0)
    out = np.empty(m, dtype=float)
    for rank, idx in enumerate(order):
        out[idx] = adjusted_sorted[rank]
    return out


def paired_rank_biserial(x, y):
    """Effect size aligned with paired Wilcoxon.
    Positive => x tends to exceed y; negative => y tends to exceed x.
    """
    d = np.asarray(x, dtype=float) - np.asarray(y, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 0.0
    absd = np.abs(d)
    ranks = pd.Series(absd).rank(method='average').to_numpy()
    w_plus = ranks[d > 0].sum()
    w_minus = ranks[d < 0].sum()
    denom = n * (n + 1) / 2
    return float((w_plus - w_minus) / denom)


def cliffs_delta(x, y):
    """Distributional effect size; ignores pairing. Included only as complementary diagnostic."""
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    y = np.asarray(pd.Series(y).dropna(), dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = 0
    lt = 0
    for a in x:
        gt += np.sum(a > y)
        lt += np.sum(a < y)
    return float((gt - lt) / (len(x) * len(y)))


def effect_label(v):
    a = abs(v)
    if a < 0.147:
        return 'negligible'
    if a < 0.33:
        return 'small'
    if a < 0.474:
        return 'medium'
    return 'large'

# -----------------------------
# 1) Auditoria de P9
# -----------------------------
counts = P9.groupby('model')['seed'].nunique()
if counts.nunique() != 1 or counts.iloc[0] != len(SEEDS):
    warnings.warn('No todos los modelos GNN tienen exactamente el mismo conjunto de semillas.')

seed_sets = {m: set(P9.loc[P9.model == m, 'seed'].astype(int)) for m in MODELS_GNN}
paired_seed_ok = len({tuple(sorted(v)) for v in seed_sets.values()}) == 1

# -----------------------------
# 2) Descriptivos GNN por 20 o 32 semillas
# -----------------------------
rows = []
for model, g in P9.groupby('model'):
    for metric in METRICS_GNN:
        vals = g[metric].dropna().astype(float)
        lo, hi = bootstrap_mean_ci(vals, seed=20261009 + sum(map(ord, model+metric)))
        rows.append({
            'family': 'GNN',
            'model': model,
            'metric': metric,
            'n_runs': len(vals),
            'mean': vals.mean(),
            'median': vals.median(),
            'std': vals.std(ddof=1),
            'iqr': iqr(vals),
            'min': vals.min(),
            'max': vals.max(),
            'ci95_mean_low_bootstrap': lo,
            'ci95_mean_high_bootstrap': hi,
        })

gnn_desc = pd.DataFrame(rows)
gnn_desc.to_csv(OUT_DIR / 'P10_GNN_descriptive_stats.csv', index=False)

# -----------------------------
# 3) Wilcoxon pareado entre GNNs por semilla
#    Holm por metrica entre las 6 comparaciones de modelos
# -----------------------------
pair_rows = []
for metric in METRICS_GNN:
    tmp_rows = []
    for a, b in combinations(MODELS_GNN, 2):
        da = P9.loc[P9.model == a, ['seed', metric]].rename(columns={metric: 'a'})
        db = P9.loc[P9.model == b, ['seed', metric]].rename(columns={metric: 'b'})
        z = da.merge(db, on='seed', how='inner').dropna()
        diff = z['a'] - z['b']
        nonzero = int((diff != 0).sum())
        if len(z) == 0:
            stat = p = np.nan
        elif nonzero == 0:
            stat, p = 0.0, 1.0
        else:
            stat, p = wilcoxon(z['a'], z['b'], alternative='two-sided', zero_method='wilcox', method='auto')
        rbc = paired_rank_biserial(z['a'], z['b']) if len(z) else np.nan
        cd = cliffs_delta(z['a'], z['b']) if len(z) else np.nan
        tmp_rows.append({
            'metric': metric,
            'model_A': a,
            'model_B': b,
            'n_paired_seeds': len(z),
            'mean_A': z['a'].mean() if len(z) else np.nan,
            'mean_B': z['b'].mean() if len(z) else np.nan,
            'mean_difference_A_minus_B': diff.mean() if len(z) else np.nan,
            'median_difference_A_minus_B': diff.median() if len(z) else np.nan,
            'wilcoxon_statistic': stat,
            'p_raw': p,
            'paired_rank_biserial': rbc,
            'paired_effect_magnitude': effect_label(rbc) if np.isfinite(rbc) else '',
            'cliffs_delta_unpaired_diagnostic': cd,
            'cliffs_effect_magnitude': effect_label(cd) if np.isfinite(cd) else '',
        })
    pvals = [r['p_raw'] for r in tmp_rows]
    adj = holm_adjust(pvals)
    for r, pa in zip(tmp_rows, adj):
        r['p_holm'] = pa
        r['significant_holm_0_05'] = bool(pa < 0.05)
        pair_rows.append(r)

gnn_pairs = pd.DataFrame(pair_rows)
gnn_pairs.to_csv(OUT_DIR / 'P10_GNN_pairwise_wilcoxon.csv', index=False)

# -----------------------------
# 4) Resultados clasicos: DESCRIPTIVOS solamente
#    P7 = una evaluacion final por modelo y escenario.
#    P8 = una evaluacion final por modelo y regimen de negativos.
# -----------------------------
classical_rows = []
for _, r in P7.iterrows():
    classical_rows.append({
        'source': 'P7-1',
        'experiment': 'class_imbalance',
        'scenario': str(r['scenario']),
        'model': r['model'],
        'auc_roc': r['auc_roc_test'],
        'auc_pr': r['auc_pr_test'],
        'f1': r['f1_test'],
        'accuracy': r['accuracy_test'],
        'balanced_accuracy': r['balanced_accuracy_test'],
        'precision': r['precision_test'],
        'recall': r['recall_test'],
        'positive_prevalence': r['positive_prevalence_test'],
        'auc_pr_random_baseline': r['auc_pr_random_baseline'],
        'inferential_unit': 'single_final_test',
    })
for _, r in P8.iterrows():
    classical_rows.append({
        'source': 'P8-1',
        'experiment': 'negative_sampling',
        'scenario': str(r['negative_mode']),
        'model': r['model'],
        'auc_roc': r['auc_roc_test'],
        'auc_pr': r['auc_pr_test'],
        'f1': r['f1_test'],
        'accuracy': r['accuracy_test'],
        'balanced_accuracy': r['balanced_accuracy_test'],
        'precision': r['precision_test'],
        'recall': r['recall_test'],
        'positive_prevalence': r['positive_prevalence_test'],
        'auc_pr_random_baseline': r['auc_pr_random_baseline'],
        'inferential_unit': 'single_final_test',
    })
classical = pd.DataFrame(classical_rows)
classical.to_csv(OUT_DIR / 'P10_classical_descriptive_results.csv', index=False)

# -----------------------------
# 5) Tabla compacta de mejores resultados por escenario
# -----------------------------
best_rows = []
for (exp, scen), g in classical.groupby(['experiment', 'scenario']):
    for metric in ['auc_pr', 'auc_roc', 'f1', 'balanced_accuracy']:
        k = g[metric].astype(float).idxmax()
        r = g.loc[k]
        best_rows.append({
            'family': 'Classical ML', 'experiment': exp, 'scenario': scen,
            'metric': metric, 'best_model': r['model'], 'best_value': r[metric],
            'n_independent_test_runs': 1,
            'inference_allowed': False,
        })

for metric in ['auc_pr', 'auc_roc', 'f1', 'balanced_accuracy']:
    d = gnn_desc[gnn_desc.metric == metric].copy()
    k = d['mean'].idxmax()
    r = d.loc[k]
    best_rows.append({
        'family': 'GNN', 'experiment': 'temporal_multiseed', 'scenario': '2023-2024',
        'metric': metric, 'best_model': r['model'], 'best_value': r['mean'],
        'n_independent_test_runs': int(r['n_runs']),
        'inference_allowed': True,
    })

best = pd.DataFrame(best_rows)
best.to_csv(OUT_DIR / 'P10_best_models_by_experiment.csv', index=False)

# -----------------------------
# 6) Auditoria metodologica
# -----------------------------
audit = pd.DataFrame([
    ['p9_models', ', '.join(MODELS_GNN)],
    ['p9_seed_min', min(SEEDS)],
    ['p9_seed_max', max(SEEDS)],
    ['p9_n_unique_seeds', len(SEEDS)],
    ['p9_same_seed_set_all_models', paired_seed_ok],
    ['p9_inferential_unit', 'seed-level final-test run'],
    ['p9_wilcoxon_paired', True],
    ['p9_multiple_testing_correction', 'Holm within each metric across model pairs'],
    ['p9_primary_paired_effect_size', 'paired rank-biserial correlation'],
    ['p9_cliffs_delta_role', 'complementary unpaired distributional diagnostic only'],
    ['p9_ci95', 'bootstrap CI of mean, 10000 resamples'],
    ['p7_inferential_tests', False],
    ['p8_inferential_tests', False],
    ['p7_p8_reason', 'single final test per model/scenario; CV folds are not treated as independent experimental runs'],
    ['cv_folds_counted_as_independent_runs', False],
    ['non_significant_means_equivalent', False],
])
audit.columns = ['item', 'value']
audit.to_csv(OUT_DIR / 'P10_audit.csv', index=False)

# -----------------------------
# 7) Resumen en consola
# -----------------------------
print('P10 completado.')
print('Directorio:', OUT_DIR)
print('\nGNN: 20 semillas validas para inferencia pareada:', paired_seed_ok)
print('\nMejores GNN por media:')
print(best[best.family == 'GNN'][['metric','best_model','best_value']].to_string(index=False))
print('\nComparaciones GNN significativas tras Holm (si existen):')
sig = gnn_pairs[gnn_pairs.significant_holm_0_05]
print(sig[['metric','model_A','model_B','p_raw','p_holm','paired_rank_biserial']].to_string(index=False) if len(sig) else 'Ninguna.')
print('\nIMPORTANTE: P7/P8 se reportan descriptivamente; no se aplican Wilcoxon usando folds de CV como replicas independientes.')


P10 completado.
Directorio: C:\Users\Dra. Pilar\UltimaversionProgramasComparacion\P10_estadistica_consolidada

GNN: 20 semillas validas para inferencia pareada: True

Mejores GNN por media:
           metric best_model  best_value
           auc_pr        GIN    0.584238
          auc_roc  GraphSAGE    0.551170
               f1  GraphSAGE    0.437691
balanced_accuracy        GIN    0.551618

Comparaciones GNN significativas tras Holm (si existen):
 metric model_A   model_B    p_raw   p_holm  paired_rank_biserial
auc_roc     GAT GraphSAGE 0.000306 0.001838             -0.696970
 auc_pr     GAT GraphSAGE 0.005712 0.034272             -0.549242
     f1     GAT GraphSAGE 0.001144 0.006866             -0.636364

IMPORTANTE: P7/P8 se reportan descriptivamente; no se aplican Wilcoxon usando folds de CV como replicas independientes.
